[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/03_grpo_mcq_lora.ipynb)

# 03 · 进阶：选择题 GRPO（RLVR）/ Advanced: GRPO on multiple-choice science

用**可验证奖励**做强化学习：模型对每道题采样多次，答对得分，组内做相对优势（GRPO，见课程第 6 讲 / RLHF Book 第 6 章）。

- 默认题库：`allenai/sciq`（公开的科学选择题，比 GPQA 简单得多，不在 GPQA 里）。也可以换成 Dolci-Think-RL / Dolci-Instruct-RL 里有标准答案的题。
- Prompt 格式**和 lm-eval 的 GPQA CoT 模板一致**，所以 RL 同时教会模型"按评测要求的格式作答"。
- 奖励：答案正确 = 1.0；出现 `The answer is (X)` 格式 = 0.2。
- 起点：推荐用 `02` 训好的 SFT 模型；也可以直接从 Base 开始（更难）。

T4 上 rollout 很慢，这里的配置偏保守（每步 2 道题 × 4 个采样，最长 768 token）。别忘了训练总时间 ≤ 4 小时包括 SFT + RL。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆比赛仓库（脚本都在里面） / clone the hackathon repo
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval

In [ ]:
# 训练环境（Unsloth + TRL + PEFT）。不要和评测在同一个 runtime 里装：vLLM 与 Unsloth 依赖的 torch 版本可能不同。
!pip install -q -r requirements-train.txt
import importlib.metadata as md
for p in ["unsloth", "trl", "peft", "transformers", "torch"]:
    print(p, md.version(p))

In [ ]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款，
# 再用有 read（提交模型需要 write）权限的 token 登录。
# 推荐：Colab 左侧 🔑 Secrets 里添加 HF_TOKEN，这里就会自动读取。
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ==== 配置 / config ====
SMOKE       = False
INIT_MODEL  = "my-team/qwen35-4b-dolci-sft"   # 02 的产出；或 "Qwen/Qwen3.5-4B-Base"
MAX_SEQ_LEN = 2048
LORA_R      = 16
N_PROMPTS   = 64 if SMOKE else 2000
MAX_STEPS   = 10 if SMOKE else 300
USE_UNSLOTH = True
HUB_REPO    = "my-team/qwen35-4b-sft-grpo"
PUSH        = not SMOKE

## 1. 构建题库（格式与 GPQA 评测模板一致）

In [ ]:
import random, re
from datasets import load_dataset

TEMPLATE = ("What is the correct answer to this question:{q}\nChoices:\n"
            "(A) {a}\n(B) {b}\n(C) {c}\n(D) {d}\nLet's think step by step: ")

def to_mcq(ex, idx):
    rng = random.Random(idx)
    choices = [ex["distractor1"], ex["distractor2"], ex["distractor3"], ex["correct_answer"]]
    rng.shuffle(choices)
    letter = "ABCD"[choices.index(ex["correct_answer"])]
    text = TEMPLATE.format(q=ex["question"], a=choices[0], b=choices[1], c=choices[2], d=choices[3])
    return {"prompt": [{"role": "user", "content": text}], "answer": letter}

raw = load_dataset("allenai/sciq", split="train").shuffle(seed=0).select(range(N_PROMPTS))
ds = raw.map(to_mcq, with_indices=True, remove_columns=raw.column_names)
print(ds[0]["prompt"][0]["content"]); print("answer:", ds[0]["answer"])

In [ ]:
# 去污染（SciQ 与 GPQA 理论上无重叠，但流程要走一遍）
ds.to_json("data/grpo_prompts.jsonl")
!python scripts/decontam.py --train-jsonl data/grpo_prompts.jsonl --out data/decontam_grpo.json

## 2. 奖励函数

In [ ]:
LETTER_RE = re.compile(r"\(([A-D])\)")
STRICT_RE = re.compile(r"The answer is \(([A-D])\)")

def _text(c):
    return c[0]["content"] if isinstance(c, list) else c

def correctness_reward(completions, answer, **kwargs):
    # 与 lm-eval flexible-extract 一致：取最后一个 (A)-(D)
    out = []
    for c, a in zip(completions, answer):
        found = LETTER_RE.findall(_text(c))
        out.append(1.0 if found and found[-1] == a else 0.0)
    return out

def format_reward(completions, **kwargs):
    return [0.2 if STRICT_RE.search(_text(c)) else 0.0 for c in completions]

print(correctness_reward([[{"content": "... The answer is (B)."}]], ["B"]), format_reward(["The answer is (C)."]))

## 3. 加载模型 + LoRA

In [ ]:
# 加载模型 + LoRA。默认走 Unsloth（更快、更省显存）；如果 Unsloth 在你的环境里报错，把 USE_UNSLOTH 改成 False，
# 会回退到 transformers + PEFT（慢一些，但依赖更少）。
# 注意：T4 没有 bf16，一律 fp16。Unsloth 官方不建议对 Qwen3.5 用 4bit QLoRA。
import torch

if USE_UNSLOTH:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name=INIT_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        full_finetuning=False,
    )
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,      # Qwen3.5 自带视觉编码器，这里只训语言部分
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=2 * LORA_R,
        lora_dropout=0,
        bias="none",
        random_state=0,
        use_gradient_checkpointing="unsloth",
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(INIT_MODEL)
    model = AutoModelForCausalLM.from_pretrained(INIT_MODEL, dtype=torch.float16, device_map={"": 0})
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.0, task_type="CAUSAL_LM",
        target_modules="all-linear", exclude_modules=r".*visual.*",
    ))

# 多模态模型可能返回 processor；TRL 只需要文本 tokenizer
tok = getattr(tokenizer, "tokenizer", tokenizer)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model.print_trainable_parameters() if hasattr(model, "print_trainable_parameters") else None
print(f"GPU mem after load: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## 4. GRPO 训练

In [ ]:
import time, torch
from trl import GRPOTrainer, GRPOConfig

cfg = GRPOConfig(
    output_dir="outputs/grpo",
    learning_rate=2e-5,
    per_device_train_batch_size=4,     # = 每步的 completion 数；4 / num_generations = 1 道题
    gradient_accumulation_steps=2,     # -> 每次生成 8 个 completion = 2 道题 × 4 个采样
    num_generations=4,
    max_completion_length=768,
    temperature=1.0,
    beta=0.0,                          # 不加 KL（Dr.GRPO / DAPO 的常见做法）；可以试 0.01–0.04
    max_steps=MAX_STEPS,
    fp16=True, bf16=False,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    push_to_hub=PUSH,
    hub_model_id=HUB_REPO + "-ckpt",
    hub_strategy="checkpoint",
    hub_private_repo=True,
    report_to="none",
    seed=0,
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward],
    args=cfg,
    train_dataset=ds,
    processing_class=tok,
)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
print(f"train time: {(time.time() - t0) / 3600:.2f} h   peak mem: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")

看日志里的 `reward`、`rewards/correctness_reward/mean` 是否上升，`completion_length` 是否失控。
如果奖励一直为 0：起点模型可能还不会这个格式，先做 `02` 的 SFT。

## 5. 合并并推到 Hub

In [ ]:
# 合并 LoRA 并以 fp16 推到 Hub（公开仓库，排行榜要求）
if PUSH:
    if USE_UNSLOTH:
        model.push_to_hub_merged(HUB_REPO, tokenizer, save_method="merged_16bit", private=False)
    else:
        merged = model.merge_and_unload()
        merged.push_to_hub(HUB_REPO, private=False)
        tok.push_to_hub(HUB_REPO, private=False)
    print(f"pushed: https://huggingface.co/{HUB_REPO}")
    print("下一步：在一个新的 runtime 里打开 01_eval_gpqa.ipynb，MODEL 填上面的仓库名。")